<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/Week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6: Recovery Mechanisms & Logging

## 1. Introduction
Databases must remain reliable despite unpredictable failures such as **power outages**, **hardware crashes**, **OS panics**, **application bugs**, or **sudden process terminations**.  
When such events occur, recovery mechanisms and logging ensure that no data is lost and that the database remains consistent.  

We cover **Write-Ahead Logging (WAL)**, **checkpoints**, **ARIES recovery**, **undo/redo logging**, and **crash consistency** as implemented in modern DBMS.

**Goals by end of session:**
- Understand the purpose and structure of logs.
- Explain how WAL ensures durability.
- Describe ARIES phases.
- Explain checkpoints and their role in reducing recovery time.
- Apply undo/redo logging principles.
- Relate recovery to ACID — particularly **atomicity** and **durability**.

---

## 2. The Need for Recovery

### 2.1 Sources of Failure
- **Transaction Failures:** Logical errors, constraint violations, deadlocks.
- **System Crashes:** Hardware faults, power outages, OS crashes.
- **Disk Failures:** Corruption, mechanical failure.
- **Application Errors:** Bugs causing unexpected termination.

### 2.2 Recovery Goals
- **Atomicity:** All or nothing.
- **Durability:** Committed changes survive.
- **Consistency:** Valid state transitions.
- **Minimal Downtime:** Fast recovery.

Achieved by **logging every change** so it can be **undone or redone**.

---

## 3. Write-Ahead Logging (WAL)

### Definition
Before a data page is written to disk, the **log record describing the change** is written to **stable storage**.

### 3.1 WAL Protocol Rules
1. **Write Log First:** Log record must be persisted before its data page.
2. **Force Log on Commit:** All transaction log records must be flushed before acknowledging commit.

**Why?**
- If crash happens **after log but before data** → **Redo** from log.
- If crash happens **before log** → No redo → Atomicity preserved.

### 3.2 Log Record Structure
- **Transaction ID (TID)**
- **Operation Type** (INSERT, UPDATE, DELETE)
- **Page ID**
- **Old Value** (undo)
- **New Value** (redo)
- **Log Sequence Number (LSN)** — strictly increasing.

**Example:**
LSN: 105  
TID: T1  
Operation: UPDATE  
Page: P7  
Before-image: 5000  
After-image: 6000  


WAL ensures this record is on disk before page P7 is written.

---

## 4. Checkpoints

### 4.1 Purpose
- Reduce recovery time.
- Mark a point where dirty pages are flushed and committed transactions are durable.

### 4.2 Types
- **Sharp Checkpoint:** Pause transactions, flush dirty pages, write checkpoint record.
- **Fuzzy Checkpoint:** Allow transactions to continue, log active transactions + dirty pages without pausing (preferred).

---

## 5. ARIES Recovery Algorithm

### 5.1 Design Principles
- Enforces **WAL**.
- **Repeats history** during redo.
- **Logs undo actions** for crash-in-crash scenarios.

### 5.2 Phases
1. **Analysis:** Start from last checkpoint, find active transactions, build **Dirty Page Table (DPT)** and **Transaction Table**.
2. **Redo:** Reapply all log changes (committed or not) from earliest LSN in DPT.
3. **Undo:** Roll back uncommitted transactions using before-images; log **Compensation Log Records (CLRs)**.

### 5.3 Example Timeline
| Time | Event                  | WAL Action |
|------|------------------------|------------|
| t1   | T1 updates page P1     | Log before/after |
| t2   | Crash                  | Recovery starts |
| t3   | Analysis finds T1 active | DPT built |
| t4   | Redo reapplies T1 change | Restore page |
| t5   | Undo reverts T1 change  | CLR written |

---

## 6. Undo and Redo Logging

### 6.1 Undo Logging
- Stores **before-image**.
- **Rule:** Before data page write → flush undo log.
- Used to roll back **uncommitted** transactions.

### 6.2 Redo Logging
- Stores **after-image**.
- **Rule:** Before commit → flush redo log.
- Used to apply committed updates not on disk.

### 6.3 Combined
- Both before/after images stored.
- Allows both rollback and roll-forward.

---

## 7. Crash Consistency
After restart:
- All committed transactions are **durable**.
- No partial updates remain.
- Data structures valid.

---

## 8. Execution Models & Recovery

### 8.1 Steal vs. No-Steal
- **Steal:** Dirty pages can be flushed before commit → Needs UNDO.
- **No-Steal:** No dirty page flushed before commit.

### 8.2 Force vs. No-Force
- **Force:** Flush pages at commit.
- **No-Force:** Pages may remain in memory → Needs REDO.

**ARIES uses:** **Steal + No-Force** → High performance, requires **UNDO + REDO**.

---

## 9. Lab: Log Recovery Simulator
Tasks:
- Implement WAL rules in Python.
- Simulate crash and recovery.
- Show checkpoints reduce recovery time.

---

## 10. Failure Scenarios for ARIES
- **Crash after Commit, before Page Flush:** Redo applies committed change.
- **Crash during Transaction Execution:** Undo reverts uncommitted work.
- **Crash during Recovery:** CLRs resume recovery exactly.

---

## 11. Capstone Project Kick-off
- Teams: 4–6 members.
- Project: Database internals, query optimization, or transaction systems.
- Include logging & recovery strategy.

---

## 12. Summary Table
| Concept          | Purpose           | Key Points |
|------------------|-------------------|------------|
| WAL              | Ensure durability | Write log before data; force on commit |
| Checkpoint       | Speed recovery    | Log active txns + dirty pages |
| ARIES            | Full recovery     | Analysis → Redo → Undo |
| Undo Logging     | Rollback          | Before-image |
| Redo Logging     | Roll-forward      | After-image |
| Crash Consistency| Post-recovery valid state | No lost commits, no partial updates |

---

## 13. Reading
- **Silberschatz, Korth, Sudarshan**, *Database System Concepts*, Ch. 17–18.  
- **Martin Kleppmann**, *Designing Data-Intensive Applications*, Ch. 7–8.  
- **C. Mohan et al.**, *ARIES: A Transaction Recovery Method…*


In [11]:
import json
import os
import time

# ====================================================
# Week 6: Recovery Mechanisms & Logging (ARIES-style)
# ====================================================

LOG_FILE = "wal.log"
DB_FILE = "database.json"

# In-memory DB (simulated)
database = {}
last_checkpoint_lsn = None  # LSN of last checkpoint


# ---------------------------
# WAL: Append log record
# ---------------------------
def write_log(record):
    """Append log record to WAL file."""
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


# ---------------------------
# Force log to disk (fsync)
# ---------------------------
def flush_log():
    """In real DBMS, flush log buffers to disk."""
    pass  # Simulated


# ---------------------------
# Write DB page to disk
# ---------------------------
def write_page_to_disk():
    with open(DB_FILE, "w") as f:
        json.dump(database, f)


# ---------------------------
# BEGIN Transaction
# ---------------------------
def begin_transaction(tid):
    print(f"[TXN {tid}] BEGIN")
    write_log({"type": "BEGIN", "tid": tid, "lsn": time.time()})


# ---------------------------
# UPDATE operation
# ---------------------------
def update_value(tid, key, new_value):
    old_value = database.get(key, None)
    print(f"[TXN {tid}] UPDATE {key}: {old_value} -> {new_value}")
    write_log({
        "type": "UPDATE",
        "tid": tid,
        "key": key,
        "before": old_value,
        "after": new_value,
        "lsn": time.time()
    })
    database[key] = new_value  # Apply in-memory (ARIES STEAL policy)


# ---------------------------
# COMMIT Transaction
# ---------------------------
def commit_transaction(tid):
    print(f"[TXN {tid}] COMMIT")
    write_log({"type": "COMMIT", "tid": tid, "lsn": time.time()})
    flush_log()


# ---------------------------
# ABORT Transaction
# ---------------------------
def abort_transaction(tid):
    print(f"[TXN {tid}] ABORT")
    write_log({"type": "ABORT", "tid": tid, "lsn": time.time()})


# ---------------------------
# Checkpoint
# ---------------------------
def checkpoint(active_txns):
    global last_checkpoint_lsn
    last_checkpoint_lsn = time.time()
    print(f"\n=== CHECKPOINT at LSN {last_checkpoint_lsn} === Active TXNs: {active_txns}")
    write_log({"type": "CHECKPOINT", "active": list(active_txns), "lsn": last_checkpoint_lsn})


# ---------------------------
# Crash Simulation
# ---------------------------
def crash():
    print("\n*** SYSTEM CRASH! ***\n")
    database.clear()


# ---------------------------
# ARIES Recovery
# ---------------------------
def recover():
    print("\n=== ARIES RECOVERY START ===")

    committed_txns = set()
    active_txns = set()
    updates_to_redo = []
    updates_to_undo = []

    # Read WAL
    with open(LOG_FILE, "r") as f:
        log_records = [json.loads(line) for line in f]

    # ---- PHASE 1: ANALYSIS ----
    print("\n--- ANALYSIS PHASE ---")
    start_index = 0
    if last_checkpoint_lsn:
        # Find last checkpoint in WAL
        for i, rec in enumerate(log_records):
            if rec["type"] == "CHECKPOINT" and rec["lsn"] == last_checkpoint_lsn:
                active_txns = set(rec["active"])
                start_index = i
                print(f"Restart from checkpoint LSN {last_checkpoint_lsn}")
                break

    for rec in log_records[start_index:]:
        if rec["type"] == "BEGIN":
            active_txns.add(rec["tid"])
        elif rec["type"] == "COMMIT":
            committed_txns.add(rec["tid"])
            active_txns.discard(rec["tid"])
        elif rec["type"] == "ABORT":
            active_txns.discard(rec["tid"])

    print(f"Committed TXNs: {committed_txns}")
    print(f"Active TXNs at crash: {active_txns}")

    # ---- PHASE 2: REDO ----
    print("\n--- REDO PHASE ---")
    for rec in log_records[start_index:]:
        if rec["type"] == "UPDATE" and rec["tid"] in committed_txns:
            print(f"Redo {rec['tid']}: {rec['key']} -> {rec['after']}")
            database[rec["key"]] = rec["after"]

    # ---- PHASE 3: UNDO ----
    print("\n--- UNDO PHASE ---")
    for rec in reversed(log_records):
        if rec["type"] == "UPDATE" and rec["tid"] in active_txns:
            print(f"Undo {rec['tid']}: {rec['key']} -> {rec['before']}")
            database[rec["key"]] = rec["before"]

    write_page_to_disk()
    print("\n=== RECOVERY COMPLETE ===")
    print("Database after recovery:", database)


# ---------------------------
# Demo Execution
# ---------------------------
if __name__ == "__main__":
    # Clean old state
    if os.path.exists(LOG_FILE): os.remove(LOG_FILE)
    if os.path.exists(DB_FILE): os.remove(DB_FILE)

    # Initial DB state
    database["A"] = 10
    database["B"] = 20
    write_page_to_disk()

    # T1: Commit before crash
    begin_transaction("T1")
    update_value("T1", "A", 100)
    commit_transaction("T1")

    # Take checkpoint
    checkpoint({"T2"})

    # T2: Active, not committed
    begin_transaction("T2")
    update_value("T2", "B", 200)

    # Simulate crash
    crash()

    # Recover using ARIES protocol
    recover()


[TXN T1] BEGIN
[TXN T1] UPDATE A: 10 -> 100
[TXN T1] COMMIT

=== CHECKPOINT at LSN 1754393716.731052 === Active TXNs: {'T2'}
[TXN T2] BEGIN
[TXN T2] UPDATE B: 20 -> 200

*** SYSTEM CRASH! ***


=== ARIES RECOVERY START ===

--- ANALYSIS PHASE ---
Restart from checkpoint LSN 1754393716.731052
Committed TXNs: set()
Active TXNs at crash: {'T2'}

--- REDO PHASE ---

--- UNDO PHASE ---
Undo T2: B -> 20

=== RECOVERY COMPLETE ===
Database after recovery: {'B': 20}


This output shows that during recovery:

- T1’s updates were not redone because it committed before the checkpoint and its changes were already on disk (no need to redo).  
- At crash time, only T2 was active and uncommitted, so no committed transactions were found in the analysis phase.  
- Redo phase did nothing since there were no committed updates after the checkpoint.  
- Undo phase rolled back T2’s update to B, restoring it to its original value `20`.  

**Conclusion:** The ARIES process correctly restored the database to a consistent state with only committed changes.
